# Upload NYC Taxi Input Data to S3

This notebook uploads a CSV file from `input_data/` to an S3-compatible bucket under the `raw_data/` prefix.


In [ ]:
# Install necessary libraries (no requirements.txt)
import sys
import subprocess
import importlib

def _ensure(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except subprocess.CalledProcessError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pip_name])

_ensure("boto3", "boto3")
_ensure("botocore", "botocore")


In [ ]:
# Import libraries
import os
from pathlib import Path
import boto3
from botocore.client import Config


In [ ]:
# Environment and S3 configuration helpers
def _get_env(name: str) -> str:
    val = os.getenv(name, "").strip()
    if not val:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return val

def _normalize_endpoint(endpoint: str) -> str:
    endpoint = endpoint.strip()
    if not endpoint.startswith("http://") and not endpoint.startswith("https://"):
        endpoint = "http://" + endpoint
    return endpoint

def _s3_client():
    key_id = _get_env("AWS_ACCESS_KEY_ID")
    secret_key = _get_env("AWS_SECRET_ACCESS_KEY")
    region = _get_env("AWS_DEFAULT_REGION")
    endpoint = _normalize_endpoint(_get_env("AWS_S3_ENDPOINT"))

    cfg = Config(signature_version="s3v4", s3={"addressing_style": "path"})
    return boto3.client(
        "s3",
        aws_access_key_id=key_id,
        aws_secret_access_key=secret_key,
        region_name=region,
        endpoint_url=endpoint,
        config=cfg,
    )


In [ ]:
# Resolve which local CSV to upload
def _pick_first_csv(local_dir: Path) -> Path:
    csv_files = sorted(p for p in local_dir.glob("*.csv") if p.is_file())
    if not csv_files:
        raise RuntimeError(f"No CSV files found in {local_dir.resolve()}")
    return csv_files[0]


In [ ]:
# Upload input CSV to S3/raw_data
s3 = _s3_client()
bucket = _get_env("AWS_S3_BUCKET")

local_input = os.getenv("TAXI_LOCAL_INPUT_CSV", "").strip()
local_dir = Path(os.getenv("TAXI_LOCAL_INPUT_DIR", "input_data"))
raw_prefix = os.getenv("TAXI_RAW_PREFIX", "raw_data/").strip() or "raw_data/"
raw_prefix = raw_prefix.rstrip("/")

csv_path = Path(local_input) if local_input else _pick_first_csv(local_dir)
if not csv_path.exists():
    raise RuntimeError(f"Local CSV not found: {csv_path}")

s3_key = f"{raw_prefix}/{csv_path.name}"

print(f"Uploading {csv_path} -> s3://{bucket}/{s3_key}", flush=True)
s3.upload_file(str(csv_path), bucket, s3_key)

print(f"Uploaded {csv_path.name} ({csv_path.stat().st_size} bytes)", flush=True)
print(f"S3 URI: s3://{bucket}/{s3_key}", flush=True)
